In [13]:
#Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [2]:
# Load the cleaned modelling dataset
file_path = "/content/modeling_dataset.csv"

df = pd.read_csv(file_path)

# Basic information
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nLabel percentages:")
print(df["label"].value_counts(normalize=True) * 100)

Dataset shape: (17535, 2)

Columns:
['Email Text', 'label']

First 5 records:


,Email Text,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",0
1,the other side of * galicismos * * galicismo *...,0
2,re : equistar deal tickets are you still avail...,0
3,\nHello I am your hot lil horny toy.\n I am...,1
4,software at incredibly low prices ( 86 % lower...,1



Missing values:
Email Text    0
label         0
dtype: int64

Duplicate rows:
0

Label distribution:
label
0    10979
1     6556
Name: count, dtype: int64

Label percentages:
label
0    62.611919
1    37.388081
Name: proportion, dtype: float64


In [4]:
# Separate input and target
X = df["Email Text"]
y = df["label"]

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide the 30% temporary set equally
# 15% validation + 15% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", len(X_train))
print("Validation set:", len(X_val))
print("Testing set:", len(X_test))

print("\nTotal:", len(X_train) + len(X_val) + len(X_test))

Training set: 12274
Validation set: 2630
Testing set: 2631

Total: 17535


In [15]:
# View Class distribution
print("Trainning set: ")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

print("\nValidation set: ")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True) * 100)

print("\nTest set: ")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True) * 100)

Trainning set: 
label
0    7685
1    4589
Name: count, dtype: int64
label
0    62.612025
1    37.387975
Name: proportion, dtype: float64

Validation set: 
label
0    1647
1     983
Name: count, dtype: int64
label
0    62.623574
1    37.376426
Name: proportion, dtype: float64

Test set: 
label
0    1647
1     984
Name: count, dtype: int64
label
0    62.599772
1    37.400228
Name: proportion, dtype: float64


In [6]:
#Save splits
train_df = pd.DataFrame({
    "Email Text": X_train,
    "label": y_train
})

val_df = pd.DataFrame({
    "Email Text": X_val,
    "label": y_val
})

test_df = pd.DataFrame({
    "Email Text": X_test,
    "label": y_test
})

train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/validation.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)

print("Saved:")
print("train.csv")
print("validation.csv")
print("test.csv")

Saved:
train.csv
validation.csv
test.csv


In [7]:
from google.colab import files

# Download training set
files.download("/content/train.csv")

# Download validation set
files.download("/content/validation.csv")

# Download test set
files.download("/content/test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
#TF-IDF
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit ONLY on training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform validation and test data
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (12274, 50000)
Validation TF-IDF shape: (2630, 50000)
Testing TF-IDF shape: (2631, 50000)


In [16]:
#Train Logistic regrassion

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model
lr_model.fit(X_train_tfidf, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [17]:
#validation evaluation

y_val_pred = lr_model.predict(X_val_tfidf)
y_val_prob = lr_model.predict_proba(X_val_tfidf)[:, 1]

accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
auc = roc_auc_score(y_val, y_val_prob)

print("Logistic Regrassion validation result: ")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=["Safe Email", "Phishing Email"]
))

Logistic Regrassion validation result: 
Accuracy : 0.9798
Precision: 0.9844
Recall   : 0.9613
F1 Score : 0.9727
ROC-AUC  : 0.9979

Classification Report:
                precision    recall  f1-score   support

    Safe Email       0.98      0.99      0.98      1647
Phishing Email       0.98      0.96      0.97       983

      accuracy                           0.98      2630
     macro avg       0.98      0.98      0.98      2630
  weighted avg       0.98      0.98      0.98      2630

